In [1]:
import warnings
from typing import Tuple, List

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Descriptors

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

In [2]:
path = "../../1_ontology/data/chebi_dataset_train.parquet"

In [3]:
def load_and_validate_smiles(filepath: str, smiles_col: str = "SMILES") -> pd.DataFrame:
    """Ładuje dane z pliku CSV i parsuje SMILES do obiektów RDKit Mol.

    Args:
        filepath: Ścieżka do pliku CSV z danymi.
        smiles_col: Nazwa kolumny zawierającej notację SMILES.

    Returns:
        pd.DataFrame: Ramka danych rozszerzona o kolumnę 'ROMol' z prawidłowymi obiektami.
    """
    df = pd.read_parquet(filepath)
    print(f"Początkowy rozmiar zbioru: {df.shape}")

    # Próba zbudowania grafu. Jeśli RDKit zwróci None, SMILES jest uszkodzony chemicznie.
    df['ROMol'] = df[smiles_col].apply(lambda x: Chem.MolFromSmiles(str(x)))

    invalid_mask = df['ROMol'].isna()
    invalid_count = invalid_mask.sum()
    print(f"Liczba uszkodzonych ciągów SMILES: {invalid_count}")

    if invalid_count > 0:
        df = df[~invalid_mask].reset_index(drop=True)
        print(f"Zbiór po oczyszczeniu: {df.shape}")

    return df

In [4]:
df = load_and_validate_smiles(path)

Początkowy rozmiar zbioru: (33668, 502)


[16:10:06] WARNING: not removing hydrogen atom without neighbors
[16:10:06] WARNING: not removing hydrogen atom without neighbors
[16:10:06] WARNING: not removing hydrogen atom without neighbors
[16:10:06] WARNING: not removing hydrogen atom without neighbors
[16:10:06] WARNING: not removing hydrogen atom without neighbors
[16:10:06] WARNING: not removing hydrogen atom without neighbors
[16:10:06] WARNING: not removing hydrogen atom without neighbors
[16:10:06] WARNING: not removing hydrogen atom without neighbors
[16:10:06] WARNING: not removing hydrogen atom without neighbors
[16:10:06] Unusual charge on atom 0 number of radical electrons set to zero
[16:10:06] WARNING: not removing hydrogen atom without neighbors
[16:10:06] WARNING: not removing hydrogen atom without neighbors
[16:10:06] WARNING: not removing hydrogen atom without neighbors
[16:10:06] WARNING: not removing hydrogen atom without neighbors
[16:10:06] WARNING: not removing hydrogen atom without neighbors
[16:10:06] WAR

Liczba uszkodzonych ciągów SMILES: 0


[16:10:10] WARNING: not removing hydrogen atom without neighbors
[16:10:10] WARNING: not removing hydrogen atom without neighbors
[16:10:10] WARNING: not removing hydrogen atom without neighbors
[16:10:10] WARNING: not removing hydrogen atom without neighbors
[16:10:10] WARNING: not removing hydrogen atom without neighbors
[16:10:10] WARNING: not removing hydrogen atom without neighbors
[16:10:10] WARNING: not removing hydrogen atom without neighbors
[16:10:10] WARNING: not removing hydrogen atom without neighbors
[16:10:10] WARNING: not removing hydrogen atom without neighbors


In [45]:
df.shape

(33668, 504)

In [35]:
import networkx as nx
import matplotlib.pyplot as plt
import re

def parse_obo_hierarchy(file_path: str):
    """Parses OBO file to extract parent-child (is_a) relationships."""
    edges = []
    current_term = None

    with open(file_path, 'r', encoding='latin-1') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # Identify start of a new term
            if line == "[Term]":
                current_term = None

            # Get current term ID
            elif line.startswith("id:"):
                current_term = line.split("id:")[1].strip()

            # Get parent ID (is_a relation)
            elif line.startswith("is_a:"):
                parent = line.split("is_a:")[1].split("!")[0].strip()
                if current_term and parent:
                    # In OBO "is_a" means: CurrentTerm is a subset of Parent
                    # So Parent is a node higher in hierarchy
                    edges.append((parent, current_term))

    return edges

def draw_hierarchy(file_path: str, max_nodes: int = 100):
    """Visualizes the hierarchy from OBO file."""
    edges = parse_obo_hierarchy(file_path)

    # Create Directed Graph
    G = nx.DiGraph()
    G.add_edges_from(edges)

    # If the graph is too large, we take a subgraph starting from the root (class_0)
    root = "class_0"
    if root in G:
        # Get nodes reachable from root within a certain distance or just a subset
        nodes = list(nx.bfs_tree(G, root, depth_limit=4))[:max_nodes]
        subgraph = G.subgraph(nodes)
    else:
        subgraph = G.subgraph(list(G.nodes())[:max_nodes])

    plt.figure(figsize=(15, 10))

    # Use shell or kamada_kawai layout for better tree visualization without Graphviz
    pos = nx.kamada_kawai_layout(subgraph)

    nx.draw(
        subgraph, pos,
        with_labels=True,
        node_size=1000,
        node_color="skyblue",
        font_size=8,
        font_weight="bold",
        arrows=True,
        edge_color="gray",
        alpha=0.8
    )

    plt.title(f"Hierarchia ChEBI (Podgląd {len(subgraph.nodes())} węzłów)")
    plt.show()


In [37]:
with open('../../1_ontology/extras/chebi_classes.obo', 'r', encoding='latin-1') as f_in:
    data = f_in.read()

with open('chebi_classes.txt', 'w', encoding='utf-8') as f_out:
    f_out.write(data)

print("Plik został pomyślnie zapisany jako chebi_classes.txt")

Plik został pomyślnie zapisany jako chebi_classes.txt


In [41]:
import pandas as pd
import networkx as nx

def generate_hierarchy_table(file_path: str):
    nodes = []
    edges = []
    names = {}

    # 1. Parsowanie pliku
    with open(file_path, 'r', encoding='latin-1') as f:
        current_id = None
        for line in f:
            line = line.strip()
            if line.startswith("id:"):
                current_id = line.split("id:")[1].strip()
            elif line.startswith("name:"):
                names[current_id] = line.split("name:")[1].strip()
            elif line.startswith("is_a:"):
                parent_id = line.split("is_a:")[1].split("!")[0].strip()
                edges.append((parent_id, current_id))

    # 2. Budowa grafu do obliczenia głębokości (Depth)
    G = nx.DiGraph()
    G.add_edges_from(edges)

    # Obliczamy odległość każdej klasy od korzenia (class_0)
    depths = {}
    if "class_0" in G:
        depths = nx.single_source_shortest_path_length(G, "class_0")

    # 3. Tworzenie listy do tabeli
    table_data = []
    for node_id in names.keys():
        # Znajdujemy wszystkich bezpośrednich rodziców
        node_parents_ids = [edge[0] for edge in edges if edge[1] == node_id]
        node_parents_names = [names.get(p_id, "N/A") for p_id in node_parents_ids]

        table_data.append({
            "ID Klasy": node_id,
            "Nazwa Klasy": names[node_id],
            "Poziom (Depth)": depths.get(node_id, 0),
            "ID Rodziców": ", ".join(node_parents_ids),
            "Nazwy Rodziców": ", ".join(node_parents_names)
        })

    # 4. Konwersja na DataFrame i sortowanie po poziomie
    df_hierarchy = pd.DataFrame(table_data)
    df_hierarchy = df_hierarchy.sort_values(by="Poziom (Depth)")

    return df_hierarchy

# Generowanie tabeli
df_hierarchy = generate_hierarchy_table("chebi_classes.txt")

In [42]:
df_hierarchy

,ID Klasy,Nazwa Klasy,Poziom (Depth),ID Rodziców,Nazwy Rodziców
0,class_0,chemical entity,0,,
1,class_1,molecular entity,1,class_0,chemical entity
417,class_474,chemical substance,1,class_0,chemical entity
297,class_366,atom,1,class_0,chemical entity
425,class_481,metal atom,2,class_366,atom
...,...,...,...,...,...
95,class_184,glucoside,12,class_161,hexoside
273,class_344,ribose phosphate,12,class_293,aldopentose phosphate
111,class_199,beta-glucoside,13,class_184,glucoside
96,class_185,D-glucoside,13,class_184,glucoside


In [40]:
import logging
from typing import List, Any, Union

import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from skfp.fingerprints import ECFPFingerprint

# Konfiguracja logowania do śledzenia przepływu
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")


class MolecularStandardizer(BaseEstimator, TransformerMixin):
    """Transformator scikit-learn do parsowania i standaryzacji struktur molekularnych.

    Klasa hermetyzuje operacje RDKit. Konwertuje ciągi SMILES na obiekty grafowe,
    a następnie przeprowadza proces odsalania (wybór największego fragmentu)
    oraz neutralizacji ładunków, co jest krytyczne dla czystości wektorów cech.
    """

    def __init__(self) -> None:
        """Inicjalizuje moduły standaryzujące z biblioteki RDKit."""
        self.fragment_chooser = rdMolStandardize.LargestFragmentChooser()
        self.uncharger = rdMolStandardize.Uncharger()

    def fit(self, X: List[str], y: Any = None) -> "MolecularStandardizer":
        """Metoda fit dla kompatybilności z interfejsem scikit-learn.

        Args:
            X: Lista wejściowych ciągów SMILES.
            y: Etykiety (ignorowane).

        Returns:
            Instancja samego siebie.
        """
        return self

    def transform(self, X: List[str]) -> List[Chem.Mol]:
        """Konwertuje i standaryzuje listę ciągów SMILES.

        Args:
            X: Lista ciągów SMILES.

        Returns:
            List[Chem.Mol]: Lista zstandaryzowanych obiektów RDKit Mol.

        Raises:
            ValueError: W przypadku błędu parsowania, uniemożliwiającego
            dalsze generowanie cech dla danej próbki.
        """
        standardized_mols = []
        for smiles in X:
            mol = Chem.MolFromSmiles(str(smiles))
            if mol is None:
                raise ValueError(
                    f"Krytyczny błąd parsowania SMILES: '{smiles}'. "
                    f"Zbiór danych musi być oczyszczony przed rurociągiem."
                )

            try:
                # Odsalanie (usunięcie np. jonów [Na+], [Cl-])
                mol = self.fragment_chooser.choose(mol)
                # Neutralizacja ładunków dla spójnej reprezentacji
                mol = self.uncharger.uncharge(mol)
            except Exception as e:
                logging.debug(f"Błąd standaryzacji dla {smiles}: {e}. Używam oryginału.")

            standardized_mols.append(mol)

        return standardized_mols


def build_feature_pipeline(fp_size: int = 2048, radius: int = 2) -> Pipeline:
    """Buduje zintegrowany rurociąg do generowania cech chemicznych.

    Wykorzystuje zliczeniową wersję Extended-Connectivity Fingerprints (ECFP),
    która zachowuje informacje o wielokrotności występowania podstruktur.

    Args:
        fp_size: Wymiarowość wektora cech (liczba wygenerowanych kolumn).
        radius: Promień poszukiwań podstruktur (radius=2 odpowiada ECFP4).

    Returns:
        Pipeline: Skonfigurowany rurociąg scikit-learn.
    """
    return Pipeline([
        ("standardizer", MolecularStandardizer()),
        ("feature_extractor", ECFPFingerprint(
            radius=radius,
            fp_size=fp_size,
            count=True,
            n_jobs=-1
        ))
    ])


def create_engineered_dataset(df: pd.DataFrame, smiles_col: str = "SMILES", fp_size: int = 2048) -> pd.DataFrame:
    """Uruchamia rurociąg i scala oryginalne dane z wygenerowanymi cechami.

    Args:
        df: Bazowa ramka danych Pandas.
        smiles_col: Nazwa kolumny zawierającej notację SMILES.
        fp_size: Rozmiar wektora fingerprintu.

    Returns:
        pd.DataFrame: Złączona ramka danych zawierająca wejściowe kolumny
        oraz nową macierz cech.
    """
    logging.info(f"Rozpoczynam ekstrakcję cech dla {len(df)} rekordów...")

    # Inicjalizacja rurociągu
    pipeline = build_feature_pipeline(fp_size=fp_size)

    # Wyciągnięcie wektorów cech (X) jako rzadka/gęsta macierz numpy
    X_smiles = df[smiles_col].tolist()
    X_features = pipeline.transform(X_smiles)

    # Konwersja macierzy do ramki danych z odpowiednimi nazwami kolumn
    feature_columns = [f"ECFP_{i}" for i in range(X_features.shape[1])]
    df_features = pd.DataFrame(X_features, columns=feature_columns, index=df.index)

    # Połączenie cech z oryginalną ramką danych
    df_final = pd.concat([df, df_features], axis=1)

    logging.info(f"Ekstrakcja zakończona pomyślnie. Wymiary nowej macierzy: {df_final.shape}")
    return df_final


In [8]:
df_preprocessed = create_engineered_dataset(df)

INFO: Rozpoczynam ekstrakcję cech dla 33668 rekordów...
[16:10:37] Running LargestFragmentChooser
[16:10:37] Running Uncharger
[16:10:37] Running LargestFragmentChooser
[16:10:37] Running Uncharger
[16:10:37] Running LargestFragmentChooser
[16:10:37] Running Uncharger
[16:10:37] Running LargestFragmentChooser
[16:10:37] Running Uncharger
[16:10:37] Running LargestFragmentChooser
[16:10:37] Running Uncharger
[16:10:37] Running LargestFragmentChooser
[16:10:37] Running Uncharger
[16:10:37] Removed negative charge.
[16:10:37] Removed negative charge.
[16:10:37] Running LargestFragmentChooser
[16:10:37] Running Uncharger
[16:10:37] Running LargestFragmentChooser
[16:10:37] Running Uncharger
[16:10:37] Running LargestFragmentChooser
[16:10:37] Running Uncharger
[16:10:37] Running LargestFragmentChooser
[16:10:37] Running Uncharger
[16:10:37] Running LargestFragmentChooser
[16:10:37] Running Uncharger
[16:10:37] Running LargestFragmentChooser
[16:10:37] Running Uncharger
[16:10:37] Running L

In [12]:
import logging
from typing import List, Any

import lightgbm as lgb
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline

from skfp.fingerprints import ECFPFingerprint
from skfp.metrics import extract_pos_proba, multioutput_auprc_score

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")


class MolecularStandardizer(BaseEstimator, TransformerMixin):
    """Transformator scikit-learn do parsowania i standaryzacji struktur molekularnych."""

    def __init__(self) -> None:
        self.fragment_chooser = rdMolStandardize.LargestFragmentChooser()
        self.uncharger = rdMolStandardize.Uncharger()

    def fit(self, X: List[str], y: Any = None) -> "MolecularStandardizer":
        return self

    def transform(self, X: List[str]) -> List[Chem.Mol]:
        standardized_mols = []
        for smiles in X:
            mol = Chem.MolFromSmiles(str(smiles))
            if mol is None:
                raise ValueError(f"Błąd parsowania SMILES: {smiles}")
            try:
                mol = self.fragment_chooser.choose(mol)
                mol = self.uncharger.uncharge(mol)
            except Exception:
                pass
            standardized_mols.append(mol)
        return standardized_mols


def build_lgbm_pipeline(fp_size: int = 2048) -> Pipeline:
    """Buduje pełny rurociąg QSAR wykorzystujący ekstraktor ECFP i model LightGBM.

    Wykorzystuje MultiOutputClassifier, aby umożliwić estymatorowi LGBM
    obsługę macierzy Y z wieloma etykietami (Extreme Multi-Label).

    Args:
        fp_size: Wymiarowość wygenerowanego wektora ECFP.

    Returns:
        Pipeline: Skonfigurowany rurociąg gotowy do treningu.
    """
    # Inicjalizacja estymatora bazowego.
    # Używamy zbalansowania klas ('is_unbalance': True), co jest kluczowe
    # dla rzadkich klas w zbiorach takich jak ChEBI.
    lgbm_base = lgb.LGBMClassifier(
        n_estimators=100,
        learning_rate=0.1,
        is_unbalance=True,
        n_jobs=-1,
        random_state=42
    )

    return Pipeline([
        ("standardizer", MolecularStandardizer()),
        ("feature_extractor", ECFPFingerprint(
            radius=2,
            fp_size=fp_size,
            count=True,
            n_jobs=-1
        )),
        ("classifier", MultiOutputClassifier(lgbm_base, n_jobs=1))
    ])


def train_and_evaluate_lgbm(
    df: pd.DataFrame,
    smiles_col: str = "SMILES",
    label_prefix: str = "class_"
) -> Pipeline:
    """Trenuje model LightGBM i ewaluuje go za pomocą metryki Multioutput AUPRC.

    Args:
        df: Ramka danych Pandas zawierająca struktury i etykiety.
        smiles_col: Nazwa kolumny z notacją SMILES.
        label_prefix: Prefiks kolumn definiujących klasy docelowe.

    Returns:
        Pipeline: Wytrenowany na danych rurociąg wdrożeniowy.
    """
    # 1. Ekstrakcja danych
    X = df[smiles_col].tolist()
    label_cols = [c for c in df.columns if str(c).startswith(label_prefix)]
    y = df[label_cols].values

    # 2. Podział zbioru (chroni przed Data Leakage na etapie Feature Engineeringu)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    logging.info(f"Rozpoczynam trening LightGBM dla {len(label_cols)} klas...")

    # 3. Inicjalizacja i trening rurociągu
    pipeline = build_lgbm_pipeline()
    pipeline.fit(X_train, y_train)

    logging.info("Trening zakończony. Generowanie predykcji prawdopodobieństw...")

    # 4. Ewaluacja dedykowana dla Multi-Label
    # predict_proba z MultiOutputClassifier zwraca listę tablic.
    y_pred_proba_list = pipeline.predict_proba(X_test)

    # Transformacja ze scikit-fingerprints - ujednolica listę do postaci macierzy (N_samples, N_classes)
    y_pred_pos = extract_pos_proba(y_pred_proba_list)

    # AUPRC to jedyna miarodajna metryka przy tak silnym niezbalansowaniu
    auprc = multioutput_auprc_score(y_test, y_pred_pos)

    print(f"\n=============================================")
    print(f" Wynik Modelu LGBM (Multioutput AUPRC): {auprc:.4f}")
    print(f"=============================================\n")

    return pipeline

In [24]:
# 1. Identyfikacja kolumn z klasami (class_0 do class_499)
class_cols = [c for c in df.columns if c.startswith('class_')]

df['n_classes'] = df[class_cols].sum(axis=1)

In [43]:
len(class_cols)

500

In [28]:
df['n_classes'].describe()

count    33668.000000
mean        27.525306
std         12.495031
min          1.000000
25%         19.000000
50%         25.000000
75%         36.000000
max         78.000000
Name: n_classes, dtype: float64

In [20]:
import logging
from typing import List

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier

from skfp.metrics import extract_pos_proba, multioutput_auprc_score

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")


def train_and_benchmark_lgbm(df: pd.DataFrame) -> MultiOutputClassifier:
    """Trains LightGBM and evaluates using Macro-F1 and AUPRC.

    Args:
        df: DataFrame with precomputed features (ECFP_*) and labels (class_*).

    Returns:
        MultiOutputClassifier: Trained model.
    """
    feature_cols = [c for c in df.columns if str(c).startswith("ECFP_")]
    label_cols = [c for c in df.columns if str(c).startswith("class_")]

    # Logi dotyczące wymiarowości danych
    logging.info(f"Liczba zmiennych opisujących rekord (features): {len(feature_cols)}")
    logging.info(f"Liczba zmiennych objaśnianych (labels): {len(label_cols)}")

    X = df[feature_cols].values
    y = df[label_cols].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    lgbm_base = lgb.LGBMClassifier(
        n_estimators=100,
        learning_rate=0.05,
        is_unbalance=True,
        n_jobs=-1,
        random_state=42
    )

    model = MultiOutputClassifier(lgbm_base, n_jobs=1)

    logging.info("Training LightGBM model...")
    model.fit(X_train, y_train)

    logging.info("Evaluating model metrics...")

    # 1. Probabilities for AUPRC
    y_prob_raw = model.predict_proba(X_test)
    y_prob_pos = extract_pos_proba(y_prob_raw)

    # 2. Hard predictions for F1 (default threshold = 0.5)
    y_pred = model.predict(X_test)

    # 3. Benchmark calculation
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    auprc = multioutput_auprc_score(y_test, y_prob_pos)

    print(f"\n--- BENCHMARK RESULTS ---")
    print(f"Macro-averaged F1 Score: {macro_f1:.4f}")
    print(f"Multioutput AUPRC:      {auprc:.4f}")
    print(f"-------------------------\n")

    return model

In [21]:
lgbm_base = train_and_benchmark_lgbm(df_preprocessed)

INFO: Liczba zmiennych opisujących rekord (features): 2048
INFO: Liczba zmiennych objaśnianych (labels): 500
INFO: Training LightGBM model...
INFO: Evaluating model metrics...



--- BENCHMARK RESULTS ---
Macro-averaged F1 Score: 0.6509
Multioutput AUPRC:      0.6630
-------------------------

